In [1]:
import os
os.chdir("..")

In [2]:
%load_ext autoreload
%autoreload 2

In [6]:
from recsys_lakehouse.spark import spark_builder
from recsys_lakehouse.jobs.lakehouse.silver import load_table
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.functions import from_unixtime, col, sum, rand, when
from pyspark.sql.types import FloatType, IntegerType, TimestampType

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("").getOrCreate()  # type: ignore

24/12/01 21:44:44 WARN Utils: Your hostname, MacBook-Pro-Milosz.local resolves to a loopback address: 127.0.0.1; using 192.168.0.73 instead (on interface en0)
24/12/01 21:44:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/01 21:44:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
books = load_table(spark,table_path=Path(".datalake/bronze/amazon_books_sample10000"), table_name="books")

In [5]:
books.show()

NameError: name 'books' is not defined

In [ ]:
books.printSchema()

In [ ]:
books.filter(F.size(col("images")) > 0).select("images").first()

In [ ]:
books_filter.select("dates").withColumn("dates", col("dates").cast(TimestampType()))

In [ ]:
books_process = books.withColumn("dates", from_unixtime(col("timestamp") / 1000, "yyyy-MM-dd HH:mm:ss")) \
     .withColumn("rating", col("rating").cast(FloatType())) \

books_filter = books_process \
    .filter(col("rating") <= 5.0) \
    .filter(col("rating") >= 1.0) \
    .filter(col("rating").isNotNull()) \
    .filter(col("asin").isNotNull()) \
    .filter(col("user_id").isNotNull()) \
    .filter(col("helpful_vote") >= 0)


books_process.printSchema()
books_process.show()
print("size: ", books_process.count())

books_filter.printSchema()
books_filter.show()
print("size: ", books_filter.count())

In [ ]:
books

In [ ]:
from recsys_lakehouse.lakehouse.silver import BooksReviewsTable

In [ ]:
b = BooksReviewsTable(books)
df = b.process(spark)

In [ ]:
b.schema

In [ ]:
df.printSchema()

In [ ]:
books.describe("helpful_vote").show()

In [ ]:
books.groupBy("rating").count().show()

In [ ]:
books_nl = books.select([
    when(rand() < 0.1, None).otherwise(col(c)).alias(c) for c in books.columns
])

In [ ]:
null_counts = books_nl.select([sum(col(c).isNull().cast("int")).alias(c) for c in books_nl.columns])
null_counts.show()

In [ ]:
meta_books = load_table(spark,table_path=Path(".datalake/bronze/amazon_books_sample10000"), table_name="meta_books")

In [ ]:
meta_books.show()

In [ ]:
null_counts = meta_books.select([sum(col(c).isNull().cast("int")).alias(c) for c in meta_books.columns])
null_counts.show()

In [ ]:
meta_books.select("average_rating").describe("average_rating").show()

In [ ]:
meta_books.filter(col("parent_asin").isNotNull()).filter(col(""))

In [ ]:
meta_books.printSchema()